In [9]:
# A) Instala o descompactador zstd exigido pelo instalador do Ollama
!sudo apt-get update && sudo apt-get install -y zstd

# B) Instala as bibliotecas Python necessárias para o projeto
!pip install ollama chromadb ipywidgets

# C) Instala o executável do Ollama no Linux do Colab
!curl -fsSL https://ollama.com/install.sh | sh

# D) Inicializa o servidor local do Ollama em segundo plano
import subprocess
import time
try:
    subprocess.Popen(["ollama", "serve"])
    time.sleep(4) # Aguarda o servidor inicializar completamente
    print("✅ Servidor Ollama iniciado com sucesso em segundo plano!")
except FileNotFoundError:
    print("❌ Erro: Executável do Ollama não encontrado.")

# E) Faz o download do modelo Llama 3.2 (1 Billion)
!ollama pull llama3.2:1b

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,410 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3bu

In [10]:
import chromadb

# Base de Conhecimento preenchida com o contexto EV ChargeOps & GoodWe
minha_base = [
    "O sistema EV ChargeOps resolve o problema de ociosidade e picos de demanda de energia em eletropostos corporativos.",
    "A potência máxima disponível no carregador rápido DC GoodWe é de 120 kW.",
    "O protocolo de comunicação nativo utilizado entre os carregadores e a plataforma EV ChargeOps é o OCPP 1.6J / 2.0.1.",
    "O sistema utiliza o algoritmo de Smart Charging para realizar o balanceamento dinâmico de carga (DLM) e evitar multas por ultrapassagem de demanda contratada.",
    "O armazenamento de energia (BESS) da GoodWe é integrado ao ChargeOps para realizar o 'peak shaving', liberando energia da bateria nos horários de pico.",
    "A plataforma EV ChargeOps monitora o status de saúde da bateria (SoH) e o estado de carga (SoC) dos veículos conectados em tempo real.",
    "O sistema de telemetria envia dados de consumo e status das estações a cada 10 segundos via conexões seguras HTTPS e MQTT.",
    "O ChargeOps mitiga falhas de conexão local utilizando uma fila de mensagens offline (arquitetura Store-and-Forward) nos carregadores.",
    "A integração com os inversores híbridos GoodWe (linha ET/ST) permite priorizar o uso de energia solar excedente para a recarga dos EVs.",
    "O sistema gera relatórios automatizados de pegada de carbono (CO2 evitado) e eficiência energética para auditorias de ESG.",
    "A autenticação dos usuários nas estações gerenciadas pelo ChargeOps pode ser feita via tags RFID, aplicativos móveis ou protocolo Plug & Charge (ISO 15118).",
    "O EV ChargeOps possui uma API RESTful para integração direta com sistemas ERP de frotas e plataformas de faturamento de energia."
]

# Inicialização do cliente ChromaDB (em memória para o Colab)
cliente = chromadb.Client()

# Criação da coleção com o nome do projeto
colecao = cliente.create_collection(name="ev_chargeops_goodwe_db")

# Indexação dos documentos na base vetorial
colecao.add(
    documents=minha_base,
    ids=[f"info_{i}" for i in range(len(minha_base))]
)

print(f"✅ {colecao.count()} informações técnicas indexadas com sucesso no ChromaDB!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 53.8MiB/s]


✅ 12 informações técnicas indexadas com sucesso no ChromaDB!


In [11]:
import ollama

perguntas_teste = [
    "Qual é o protocolo de comunicação padrão e a frequência de envio dos dados de telemetria?",
    "Como os inversores híbridos da GoodWe atuam junto com o ChargeOps na recarga de EVs?",
    "O que o sistema faz caso a estação de recarga perca a conexão com a internet local?",
    "Qual é a receita anual da Tesla com a venda de créditos de carbono nos Estados Unidos?",
    "O sistema usa inteligência artificial para prever o preço das ações da GoodWe na bolsa?",
]

print("=" * 60)
print("🧪 EXECUTANDO TESTES RAG — EV CHALLENGE 2026")
print("=" * 60)

for p in perguntas_teste:
    resultados = colecao.query(query_texts=[p], n_results=2)
    contexto = "\n".join(resultados["documents"][0])

    prompt = (
        "Diretrizes: Responda de forma estritamente técnica com base apenas no CONTEXTO fornecido. "
        "Se a resposta não puder ser deduzida do contexto ou se fugir do escopo (recarga de VEs, inversores GoodWe), "
        "responda obrigatoriamente e sem justificativas: 'Lamento, mas essa informação não consta na minha base de dados atual.'\n\n"
        f"CONTEXTO:\n{contexto}\n\n"
        f"PERGUNTA: {p}"
    )

    r = ollama.chat(model="llama3.2:1b", messages=[{"role": "user", "content": prompt}])
    print(f"\n👤 Pergunta: {p}")
    print(f"🤖 Resposta: {r['message']['content']}")
    print("─" * 60)

🧪 EXECUTANDO TESTES RAG — EV CHALLENGE 2026

👤 Pergunta: Qual é o protocolo de comunicação padrão e a frequência de envio dos dados de telemetria?
🤖 Resposta: Com base no contexto fornecido, não há informações que sugiram um protocolo de comunicação padrão específico para os dados de telemetria enviados por meio da conexão segura HTTPS e MQTT. No entanto, podemos analisar as opções de autenticação mencionadas:

1. **Tags RFID**: O uso de tags RFID pode ser uma forma de autenticação, mas não há informações que sugiram que a autenticação seja feita por meio dessas tags.
2. **Aplicativos móveis**: A possibilidade de autenticação através de aplicativos móveis é um bom ponto de partida, pois os dispositivos móveis são comuns e podem ter recursos de autenticação avançados.
3. **Protocolo Plug & Charge (ISO 15118)**: Embora o ISO 15118 seja especificamente relacionado à comunicação entre estações de recarga e cargas, não há informações que sugiram que ele seja um protocolo de comunicação comu

In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML
import ollama

# 1. Inicialização do Histórico Limpo (Apenas com a instrução de Persona)
historico = [{
    "role": "system",
    "content": (
        "Você é um assistente técnico em EV ChargeOps e GoodWe. "
        "Sua única tarefa é responder à pergunta do usuário baseando-se no contexto fornecido. "
        "Seja extremamente direto, curto e use os dados numéricos do contexto. "
        "Não repita as instruções que recebeu."
    )
}]

# 2. Definição dos Componentes Visuais (Título personalizado com o seu Nome e RM)
titulo = widgets.HTML(
    "<h3 style='font-family:sans-serif; color:#007bff; margin-bottom: 5px;'>🔌 EV ChargeOps Bot — EV Challenge 2026</h3>"
    "<p style='font-family:sans-serif; color:#666; font-size:12px; margin-top: 0px; margin-bottom: 15px;'>"
    "<b>Desenvolvedor:</b> Pedro Soares de Souza | <b>RM:</b> 571285"
    "</p>"
)

area_chat = widgets.Output(layout=widgets.Layout(
    height="320px", overflow_y="auto", border="1px solid #ddd", padding="10px", margin="0 0 10px 0"
))

campo_texto = widgets.Text(
    placeholder="Digite sua dúvida sobre recarga e infraestrutura GoodWe...", layout=widgets.Layout(width="75%")
)

botao_enviar = widgets.Button(
    description="Enviar 📨", button_style="primary", layout=widgets.Layout(width="23%")
)

botao_limpar = widgets.Button(
    description="Limpar Histórico 🗑️", button_style="warning", layout=widgets.Layout(width="100%", margin="10px 0 0 0")
)

barra = widgets.HBox([campo_texto, botao_enviar])

# 3. Lógica do Pipeline RAG com separação rígida de System/User
def ao_enviar_com_rag(b):
    pergunta = campo_texto.value.strip()
    if not pergunta:
        return
    campo_texto.value = ""

    # Balão do Usuário (Direita - Azul)
    with area_chat:
        display(HTML(
            f'<div style="text-align:right;margin:6px 0">'
            f'<span style="background:#007bff;color:white;padding:7px 13px;'
            f'border-radius:18px 18px 4px 18px;display:inline-block;font-family:sans-serif">👤 {pergunta}</span></div>'
        ))

    # Executa a busca por similaridade semântica no ChromaDB
    try:
        resultados = colecao.query(query_texts=[pergunta], n_results=2)
        contexto = "\n".join(resultados["documents"][0])
    except Exception:
        contexto = "Erro ao acessar a base do ChromaDB."

    # Se a pergunta for totalmente fora do escopo, o próprio código bloqueia antes de mandar para o LLM
    palavras_chave = ["potência", "kw", "protocolo", "ocpp", "charging", "bess", "bateria", "status", "telemetria", "conexão", "solar", "inversor", "esg", "autenticação", "api", "carregador", "vaga"]
    forca_bloqueio = not any(palavra in pergunta.lower() for palavra in palavras_chave)

    if forca_bloqueio:
        resposta = "Lamento, mas essa informação não consta na minha base de dados atual."
    else:
        # Montamos a mensagem do usuário contendo o contexto e a pergunta de forma simples
        conteudo_usuario = (
            f"CONTEXTO DO PROJETO:\n{contexto}\n\n"
            f"PERGUNTA DO USUÁRIO: {pergunta}\n\n"
            "RESPONDA APENAS A PERGUNTA USANDO O CONTEXTO:"
        )

        try:
            # Enviamos a estrutura correta: System (Persona) + User (Dados + Pergunta)
            resposta_obj = ollama.chat(
                model="llama3.2:1b",
                messages=[
                    {"role": "system", "content": historico[0]["content"]},
                    {"role": "user", "content": conteudo_usuario}
                ]
            )
            resposta = resposta_obj["message"]["content"].strip()
        except Exception as e:
            resposta = f"Erro na comunicação com o Ollama: {str(e)}"

    # Atualiza o histórico para registro da sessão
    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})

    # Balão do Assistente (Esquerda - Cinza)
    with area_chat:
        display(HTML(
            f'<div style="text-align:left;margin:6px 0">'
            f'<span style="background:#e9ecef;color:#212529;padding:7px 13px;'
            f'border-radius:18px 18px 18px 4px;display:inline-block;font-family:sans-serif">🤖 {resposta}</span></div>'
        ))

def ao_limpar_com_rag(b):
    area_chat.clear_output()
    with area_chat:
        display(HTML("<i style='color:#999;font-family:sans-serif'>Conversa reiniciada e memória limpa! ✨</i>"))

# 4. Vinculação dos Gatilhos de Eventos
botao_enviar.on_click(ao_enviar_com_rag)
campo_texto.on_submit(ao_enviar_com_rag)
botao_limpar.on_click(ao_limpar_com_rag)

# 5. Renderização Final
display(titulo, area_chat, barra, botao_limpar)

HTML(value="<h3 style='font-family:sans-serif; color:#007bff; margin-bottom: 5px;'>🔌 EV ChargeOps Bot — EV Cha…

Output(layout=Layout(border='1px solid #ddd', height='320px', margin='0 0 10px 0', overflow_y='auto', padding=…

Button(button_style='warning', description='Limpar Histórico 🗑️', layout=Layout(margin='10px 0 0 0', width='10…